In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.3 SwiGLU

The classic MLP is `Linear -> GELU -> Linear`. SwiGLU adds a multiplicative gate:
`down( silu(gate(x)) * up(x) )`. The gate lets the network modulate information
multiplicatively, which is more expressive per parameter.

In [ ]:
from pocketlm import SwiGLU
from torch import nn

dim = 16
swiglu = SwiGLU(dim)
gelu_mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
x = torch.randn(2, 5, dim)
print("SwiGLU preserves shape:", tuple(swiglu(x).shape))
print("SwiGLU params:", sum(p.numel() for p in swiglu.parameters()),
      " GELU-MLP params:", sum(p.numel() for p in gelu_mlp.parameters()))

SwiGLU uses three projections (gate, up, down) instead of two; implementations
usually shrink the hidden size to keep the budget equal. The gain is the gating,
not the parameter count.

In [ ]:
assert tuple(swiglu(x).shape) == (2, 5, dim)